# 4.3 RLHF / RLHF

用人的偏好来训练模型。推导奖励模型的损失、带 KL 约束的目标及其闭式最优解，并理解 PPO 的裁剪。

Training a model from human preferences. Derive the reward-model loss, the KL-constrained objective with its closed-form optimum, and understand the clipping of PPO.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 用成对比较训练奖励模型 / Train a reward model from pairwise comparisons

实现 Bradley-Terry 概率和成对损失，再用梯度下降训练一个线性的奖励模型。数据是模拟的，200 个“回答”各有一个隐藏的真实质量，4000 次比较按 Bradley-Terry 模型带噪声地生成。测试会检查学到的奖励在新的回答对上的排序与真实质量的一致率超过 95%。

Implement the Bradley-Terry probability and the pairwise loss, then train a linear reward model by gradient descent. The data are simulated. Each of 200 responses has a hidden true quality, and 4000 comparisons are generated with noise from the Bradley-Terry model. The tests check that the learned reward ranks fresh pairs of responses like the true quality in more than 95% of cases.

In [ ]:
import torch
import torch.nn.functional as F


def preference_prob(r_a, r_b):
    """Bradley-Terry model: the probability that a is preferred over b, given their reward scores."""
    # TODO 1: sigmoid of the difference
    raise NotImplementedError


def pairwise_loss(r_winner, r_loser):
    """Reward-model loss for a batch of comparisons: mean of  -log sigmoid(r_winner - r_loser).

    Use F.logsigmoid for numerical stability.
    """
    # TODO 2
    raise NotImplementedError


def train_reward_model(features, winners, losers, steps=300, lr=0.1):
    """Fit a linear reward model r(x) = x @ w from comparisons.

    features: (N, d) float tensor, one row per response.
    winners, losers: long tensors (M,). In comparison j, response winners[j] was preferred over losers[j].
    Start from w = 0, run plain gradient descent on pairwise_loss, return w as a (d,) tensor.
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch

assert abs(preference_prob(torch.tensor(1.0), torch.tensor(1.0)).item() - 0.5) < 1e-12, "equal scores, even odds"
assert abs(preference_prob(torch.tensor(2.0), torch.tensor(0.0)).item() - 1 / (1 + math.exp(-2))) < 1e-7
a, b = torch.tensor(0.3), torch.tensor(-1.1)
assert abs(preference_prob(a, b).item() + preference_prob(b, a).item() - 1) < 1e-7
assert abs(preference_prob(a + 5, b + 5).item() - preference_prob(a, b).item()) < 1e-7, "only the difference of scores matters"

rw, rl = torch.tensor([2.0, 0.0, -1.0]), torch.tensor([0.0, 0.0, 3.0])
expected = -sum(math.log(1 / (1 + math.exp(-(x - y)))) for x, y in zip(rw.tolist(), rl.tolist())) / 3
assert abs(pairwise_loss(rw, rl).item() - expected) < 1e-6
assert torch.isfinite(pairwise_loss(torch.tensor([-1000.0]), torch.tensor([1000.0]))), "use logsigmoid, not log(sigmoid(...))"

# Simulated annotators: a hidden "true quality" decides each comparison, with Bradley-Terry noise.
g = torch.Generator().manual_seed(0)
N, d, M = 200, 5, 4000
X = torch.randn(N, d, generator=g, dtype=torch.float64)
w_true = torch.tensor([2.0, -1.0, 0.5, 0.0, 1.5], dtype=torch.float64)
quality = X @ w_true
i = torch.randint(0, N, (M,), generator=g); j = torch.randint(0, N, (M,), generator=g)
i_wins = torch.rand(M, generator=g, dtype=torch.float64) < torch.sigmoid(quality[i] - quality[j])
winners, losers = torch.where(i_wins, i, j), torch.where(i_wins, j, i)

w = train_reward_model(X, winners, losers)
assert w.shape == (d,)
learned = X @ w
# the learned reward should rank fresh pairs the way the true quality does
a, b = torch.randint(0, N, (5000,), generator=g), torch.randint(0, N, (5000,), generator=g)
agree = ((learned[a] > learned[b]) == (quality[a] > quality[b])).double().mean().item()
assert agree > 0.95, f"ranking agreement {agree:.3f}"
cos = torch.dot(w, w_true) / (w.norm() * w_true.norm())
assert cos > 0.98, f"the learned direction should match the hidden one, cosine {cos:.3f}"
print(f"m43_reward: all tests passed (ranking agreement {agree:.3f}, cosine {cos:.3f})")
print("  learned w:", [round(x, 2) for x in w.tolist()], " hidden w:", w_true.tolist())

<details><summary><b>提示 1 / Hint 1</b></summary>

损失写成 `-F.logsigmoid(r_winner - r_loser).mean()`。先算 sigmoid 再取 log 在差值很负时会得到负无穷。

Write the loss as `-F.logsigmoid(r_winner - r_loser).mean()`. Taking the sigmoid first and then the log gives negative infinity for very negative differences.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

训练时先算出所有回答的分数 `r = features @ w`，再用 `r[winners]` 和 `r[losers]` 取出每次比较中两个回答的分数。

In training first compute the scores of all responses, `r = features @ w`, then pick the scores of the two responses of each comparison with `r[winners]` and `r[losers]`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

手动做梯度下降时，把更新放在 `with torch.no_grad():` 里，并在每步之后把 `w.grad` 清零。

For manual gradient descent put the update inside `with torch.no_grad():` and zero `w.grad` after every step.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
import torch.nn.functional as F


def preference_prob(r_a, r_b):
    return torch.sigmoid(r_a - r_b)


def pairwise_loss(r_winner, r_loser):
    return -F.logsigmoid(r_winner - r_loser).mean()


def train_reward_model(features, winners, losers, steps=300, lr=0.1):
    w = torch.zeros(features.shape[1], dtype=features.dtype, requires_grad=True)
    for _ in range(steps):
        r = features @ w                                   # one score per response
        loss = pairwise_loss(r[winners], r[losers])
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            w.grad.zero_()
    return w.detach()
```

</details>

## 2. KL 约束目标、闭式最优解与 PPO 裁剪 / The KL-constrained objective, its closed-form optimum and PPO clipping

在只有 6 个候选回答的玩具问题上实现 RLHF 的核心部件，包括带 KL 约束的目标、它的闭式最优策略、PPO 的裁剪目标，以及一步基于采样的策略梯度更新。测试会验证闭式解的目标值等于 $\beta \log Z$ 且不低于 200 个随机策略，并检查用策略梯度迭代 400 步后收敛到闭式解。

On a toy problem with only 6 candidate responses, implement the core parts of RLHF, namely the KL-constrained objective, its closed-form optimal policy, the clipped objective of PPO and one sampled policy-gradient update. The tests verify that the objective value of the closed form equals $\beta \log Z$ and is no lower than that of 200 random policies, and check that 400 policy-gradient steps converge to the closed form.

In [ ]:
import torch


def kl_regularized_objective(logp, logp_ref, rewards, beta):
    """J(pi) = E_{y ~ pi}[ r(y) ] - beta * KL(pi || pi_ref)   for one prompt with K possible responses.

    logp, logp_ref: (K,) log-probabilities of the policy and of the reference model. rewards: (K,).
    """
    # TODO 1: KL(pi || pi_ref) = sum_y pi(y) * (log pi(y) - log pi_ref(y))
    raise NotImplementedError


def optimal_policy(logp_ref, rewards, beta):
    """The closed-form maximizer of J:  pi*(y) proportional to pi_ref(y) * exp(r(y) / beta).  Returns probabilities (K,)."""
    # TODO 2
    raise NotImplementedError


def ppo_clip_objective(logp_new, logp_old, advantages, eps=0.2):
    """PPO's clipped surrogate objective (to be maximized), averaged over the batch.

    ratio = exp(logp_new - logp_old)
    objective = mean( min( ratio * A,  clip(ratio, 1 - eps, 1 + eps) * A ) )
    """
    # TODO 3
    raise NotImplementedError


def reinforce_step(theta, logp_ref, rewards, beta, lr=0.5, n_samples=512, generator=None):
    """One policy-gradient step on the KL-regularized objective. Returns the new theta (K,).

    The policy is pi = softmax(theta).
      1. sample n_samples responses y from pi           (torch.multinomial, replacement=True)
      2. per-sample return  R = r(y) - beta * (log pi(y) - log pi_ref(y)),  treated as a constant (detach)
      3. subtract the mean of R as a baseline
      4. surrogate loss = -mean( log pi(y) * (R - baseline) );  one gradient-descent step on theta
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

torch.manual_seed(0)
K, beta = 6, 0.5
logp_ref = torch.log_softmax(torch.randn(K, dtype=torch.float64), -1)
r = torch.randn(K, dtype=torch.float64)

# 1. the objective
assert abs(kl_regularized_objective(logp_ref, logp_ref, r, beta).item() - (logp_ref.exp() * r).sum().item()) < 1e-12, "KL of a distribution with itself is 0"
star = optimal_policy(logp_ref, r, beta)
assert abs(star.sum().item() - 1) < 1e-12
J_star = kl_regularized_objective(star.log(), logp_ref, r, beta).item()
Z = (logp_ref.exp() * torch.exp(r / beta)).sum()
assert abs(J_star - beta * torch.log(Z).item()) < 1e-10, "the optimal value is beta * log Z"
for _ in range(200):                                      # no other policy does better
    q = torch.log_softmax(torch.randn(K, dtype=torch.float64) * 2, -1)
    assert kl_regularized_objective(q, logp_ref, r, beta).item() <= J_star + 1e-10
# 2. beta controls how far the policy may move
assert torch.allclose(optimal_policy(logp_ref, r, 1e6), logp_ref.exp(), atol=1e-4), "huge beta: stay at the reference model"
assert optimal_policy(logp_ref, r, 1e-3).argmax() == r.argmax() and optimal_policy(logp_ref, r, 1e-3).max() > 0.999, "tiny beta: put everything on the highest reward"

# 3. PPO clipping
lo, ln = torch.log(torch.tensor([0.5, 0.5, 0.5, 0.5])), torch.log(torch.tensor([0.9, 0.9, 0.1, 0.1]))
A = torch.tensor([1.0, -1.0, 1.0, -1.0])
# ratios are 1.8, 1.8, 0.2, 0.2 -> per sample: min(1.8, 1.2)=1.2, min(-1.8, -1.2)=-1.8, min(0.2, 0.8)=0.2, min(-0.2, -0.8)=-0.8
assert abs(ppo_clip_objective(ln, lo, A, eps=0.2).item() - (1.2 - 1.8 + 0.2 - 0.8) / 4) < 1e-6
assert abs(ppo_clip_objective(lo, lo, A).item() - A.mean().item()) < 1e-6, "at ratio 1 the objective is the mean advantage"
x = ln.clone().requires_grad_(True)
ppo_clip_objective(x, lo, A).backward()
assert x.grad[0] == 0, "a good action whose probability already rose past 1 + eps gets no further push"
assert x.grad[1] != 0, "a bad action whose probability rose is still pushed back"

# 4. policy gradient finds the closed-form optimum
theta = logp_ref.clone()
g = torch.Generator().manual_seed(0)
for _ in range(400):
    theta = reinforce_step(theta, logp_ref, r, beta, generator=g)
err = (torch.softmax(theta, -1) - star).abs().max().item()
assert err < 0.03, f"after 400 steps the policy should be close to the closed form, max error {err:.3f}"
print(f"m43_objective: all tests passed (max error to the closed form {err:.4f})")
print("  reference:", [round(v, 3) for v in logp_ref.exp().tolist()])
print("  rewards  :", [round(v, 2) for v in r.tolist()])
print("  optimum  :", [round(v, 3) for v in star.tolist()])

<details><summary><b>提示 1 / Hint 1</b></summary>

闭式最优策略在对数空间里就是一行，即 `torch.softmax(logp_ref + rewards / beta, dim=-1)`。

In log space the closed-form optimum is a single line, `torch.softmax(logp_ref + rewards / beta, dim=-1)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

PPO 目标先算 `ratio = torch.exp(logp_new - logp_old)`，再对 `ratio * A` 和 `torch.clamp(ratio, 1 - eps, 1 + eps) * A` 逐元素取 `torch.min`。

For the PPO objective compute `ratio = torch.exp(logp_new - logp_old)` and take the element-wise `torch.min` of `ratio * A` and `torch.clamp(ratio, 1 - eps, 1 + eps) * A`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

策略梯度一步中，回报 `R` 要 `.detach()`，否则梯度会通过 `R` 中的 `log pi` 多算一遍。代理损失是 `-(logp[y] * (R - R.mean())).mean()`。

In the policy-gradient step the return `R` must be detached, or the gradient flows once more through the `log pi` inside `R`. The surrogate loss is `-(logp[y] * (R - R.mean())).mean()`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def kl_regularized_objective(logp, logp_ref, rewards, beta):
    p = logp.exp()
    return (p * rewards).sum() - beta * (p * (logp - logp_ref)).sum()


def optimal_policy(logp_ref, rewards, beta):
    return torch.softmax(logp_ref + rewards / beta, dim=-1)


def ppo_clip_objective(logp_new, logp_old, advantages, eps=0.2):
    ratio = torch.exp(logp_new - logp_old)
    clipped = torch.clamp(ratio, 1 - eps, 1 + eps)
    return torch.min(ratio * advantages, clipped * advantages).mean()


def reinforce_step(theta, logp_ref, rewards, beta, lr=0.5, n_samples=512, generator=None):
    theta = theta.detach().clone().requires_grad_(True)
    logp = torch.log_softmax(theta, dim=-1)
    y = torch.multinomial(logp.exp(), n_samples, replacement=True, generator=generator)
    R = (rewards[y] - beta * (logp[y] - logp_ref[y])).detach()
    loss = -(logp[y] * (R - R.mean())).mean()
    loss.backward()
    return (theta - lr * theta.grad).detach()
```

</details>

## 3. 推导带 KL 约束目标的最优策略 / Derive the optimal policy of the KL-constrained objective

这是 RLHF 和 DPO 共同的理论基础，面试中常被要求推导。合上上面的推导，在纸上完成。
(a) 从 $J(\pi) = \mathbb{E}_{y \sim \pi}[r(y)] - \beta D_{\mathrm{KL}}(\pi \| \pi_{\text{ref}})$ 出发，证明 $J(\pi) = -\beta D_{\mathrm{KL}}(\pi \| \pi^{*}) + \beta \log Z$，并写出 $\pi^{*}$ 和 $Z$。
(b) 说明 $\beta \to 0$ 和 $\beta \to \infty$ 时 $\pi^{*}$ 分别趋向什么。
(c) 由 Bradley-Terry 模型 $P(a \succ b) = \sigma(r_a - r_b)$ 出发，写出一组比较数据的负对数似然。
(d) PPO 中取 $\epsilon = 0.2$。某个回答的概率之比为 $\rho = 1.5$。分别计算 $A = 2$ 和 $A = -2$ 时该样本对裁剪目标的贡献，并解释两者的区别。

This is the shared theoretical basis of RLHF and DPO and is often asked for in interviews. Close the derivation above and work on paper.
(a) Starting from $J(\pi) = \mathbb{E}_{y \sim \pi}[r(y)] - \beta D_{\mathrm{KL}}(\pi \| \pi_{\text{ref}})$, show that $J(\pi) = -\beta D_{\mathrm{KL}}(\pi \| \pi^{*}) + \beta \log Z$ and write out $\pi^{*}$ and $Z$.
(b) State what $\pi^{*}$ tends to as $\beta \to 0$ and as $\beta \to \infty$.
(c) Starting from the Bradley-Terry model $P(a \succ b) = \sigma(r_a - r_b)$, write the negative log-likelihood of a set of comparisons.
(d) Take $\epsilon = 0.2$ in PPO. The probability ratio of some response is $\rho = 1.5$. Compute the contribution of this sample to the clipped objective for $A = 2$ and for $A = -2$, and explain the difference.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 把 $r(y)$ 写成 $\beta \log e^{r(y)/\beta}$，与 KL 中的对数合并。

(a) Write $r(y)$ as $\beta \log e^{r(y)/\beta}$ and merge it with the log in the KL.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 分别计算 $\rho A$ 和 $\mathrm{clip}(\rho, 0.8, 1.2) A$，取较小的一个。

(d) Compute $\rho A$ and $\mathrm{clip}(\rho, 0.8, 1.2) A$ and take the smaller.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $$J(\pi) = \sum_y \pi(y)\Big[r(y) - \beta \log\frac{\pi(y)}{\pi_{\text{ref}}(y)}\Big] = -\beta \sum_y \pi(y) \log\frac{\pi(y)}{\pi_{\text{ref}}(y)\, e^{r(y)/\beta}}$$ 令 $Z = \sum_y \pi_{\text{ref}}(y) e^{r(y)/\beta}$，$\pi^{*}(y) = \pi_{\text{ref}}(y) e^{r(y)/\beta} / Z$，则分母等于 $Z \pi^{*}(y)$，于是 $$J(\pi) = -\beta \sum_y \pi(y) \log\frac{\pi(y)}{\pi^{*}(y)} + \beta \log Z = -\beta D_{\mathrm{KL}}(\pi \| \pi^{*}) + \beta \log Z$$ KL 非负，所以 $\pi = \pi^{*}$ 时 $J$ 取最大值 $\beta \log Z$。

(b) $\beta \to 0$ 时 $e^{r/\beta}$ 中奖励最高的一项占绝对优势，$\pi^{*}$ 把全部概率放在奖励最高的回答上。$\beta \to \infty$ 时 $e^{r/\beta} \to 1$，$\pi^{*} \to \pi_{\text{ref}}$。

(c) $\mathcal{L} = -\sum_{j} \log \sigma\big(r_{w_j} - r_{l_j}\big)$，其中 $w_j$ 和 $l_j$ 是第 $j$ 次比较中被偏好的和未被偏好的回答。

(d) $A = 2$ 时，$\rho A = 3$，裁剪后为 $1.2 \times 2 = 2.4$，取较小值 $2.4$。这个好回答的概率已经提高了 50%，超出上限的部分不再计入，梯度为 0。$A = -2$ 时，$\rho A = -3$，裁剪后为 $-2.4$，取较小值 $-3$。这个差回答的概率反而被提高了，没有被裁剪，完整的惩罚会把它拉回来。

---

(a) $$J(\pi) = \sum_y \pi(y)\Big[r(y) - \beta \log\frac{\pi(y)}{\pi_{\text{ref}}(y)}\Big] = -\beta \sum_y \pi(y) \log\frac{\pi(y)}{\pi_{\text{ref}}(y)\, e^{r(y)/\beta}}$$ With $Z = \sum_y \pi_{\text{ref}}(y) e^{r(y)/\beta}$ and $\pi^{*}(y) = \pi_{\text{ref}}(y) e^{r(y)/\beta} / Z$ the denominator equals $Z \pi^{*}(y)$, so $$J(\pi) = -\beta \sum_y \pi(y) \log\frac{\pi(y)}{\pi^{*}(y)} + \beta \log Z = -\beta D_{\mathrm{KL}}(\pi \| \pi^{*}) + \beta \log Z$$ KL is non-negative, so $J$ reaches its maximum $\beta \log Z$ at $\pi = \pi^{*}$.

(b) As $\beta \to 0$ the term with the highest reward dominates $e^{r/\beta}$ and $\pi^{*}$ puts all probability on the response with the highest reward. As $\beta \to \infty$, $e^{r/\beta} \to 1$ and $\pi^{*} \to \pi_{\text{ref}}$.

(c) $\mathcal{L} = -\sum_{j} \log \sigma\big(r_{w_j} - r_{l_j}\big)$, where $w_j$ and $l_j$ are the preferred and the other response of comparison $j$.

(d) For $A = 2$, $\rho A = 3$ and the clipped value is $1.2 \times 2 = 2.4$, so the smaller is $2.4$. The probability of this good response has already risen by 50%, the part beyond the bound no longer counts and the gradient is 0. For $A = -2$, $\rho A = -3$ and the clipped value is $-2.4$, so the smaller is $-3$. The probability of this poor response was raised, nothing is clipped and the full penalty pulls it back.

</details>